# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [ ]:
# Load secrets file variables into the environment
%load_ext dotenv 
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [ ]:
import sys, os
# Let Python find modules in the 05_src folder
sys.path.append('../05_src/')
from utils.clients import get_client


# Build the API client using the API key and base URL from the environment variables
client = get_client()
# Constant. Pick model from env var. Default is gpt-4o-mini 
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# PDF to load and summarize: AI Report (option #2)
PDF_URL = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
# Loader object to read the PDF and store it in loader 
loader = PyPDFLoader(PDF_URL)
# Download the PDF and load it into a list of Document objects (one per page)
docs = loader.load()

# Concatenate the text of all pages into a single string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# Total number of pages and characters in the document
print(len(docs), "pages,", len(document_text), "characters")

26 pages, 53851 characters


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
# Output object: Pydantic basemodel object to hold the summary (what AI says)
class SummaryContent(BaseModel):
    Author: str
    Title: str
    Relevance: str   # one paragraph on why it matters to an AI professional
    Summary: str     # no longer than 1000 tokens
    Tone: str

# Input and output token counts for the summary (what happened during the AI call)
class ArticleSummary(SummaryContent):
    InputTokens: int
    OutputTokens: int

In [ ]:
# Constant
TONE = "Victorian English"

# System instructions: define the model's role and the rules it must follow
instructions = (
    "You are an expert technical editor who summarises professional articles "
    "for an audience of AI practitioners.\n"
    "Follow these rules strictly:\n"
    # Rule 1: Summary and Relevance in the specified tone
    "1. Write the Summary and Relevance entirely in {TONE}: formal, "
    "period-appropriate vocabulary and elaborate but grammatical sentences.\n"
    # Rule 2: stay faithful to the source and cap the length
    "2. The Summary must be faithful to the source (invent no facts) and no "
    "longer than 1000 tokens.\n"
    # Rule 3 — keep Relevance to a single paragraph
    "3. Relevance is at most one paragraph on why an AI professional should care.\n"
    # Rule 4 — pull Author and Title from the text, or infer if missing
    "4. Extract Author and Title from the document; if absent, infer them.\n"
    # Rule 5 — record the tone used, matching the TONE constant exactly
    "5. Set Tone to exactly: {TONE}."
)

# User message: hand the model the article text with clear fences
user_prompt = (
    "Summarise the following article according to your instructions.\n\n"
    "=== ARTICLE START ===\n"
    "{document_text}\n"
    "=== ARTICLE END ==="
)

In [ ]:
# Call the AI model. Pass system instructions + user prompt. Require the reply to match the SummaryContent schema (structured output)
response = client.responses.parse(
    model=MODEL,
    instructions=instructions,
    input=[{"role": "user", "content": user_prompt}],
    text_format=SummaryContent,
)
# The validated SummaryContent output from the model
content = response.output_parsed

# Build the full summary object with the summary from the model and the tokens used from the call.
summary_obj = ArticleSummary(
    **content.model_dump(),
    InputTokens=response.usage.input_tokens,        # tokens consumed by the prompt
    OutputTokens=response.usage.output_tokens,      # tokens produced in the response
)
# full summary object
summary_obj

ArticleSummary(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='In this ever-evolving landscape of generative AI, AI practitioners must remain vigilant regarding the stark divide unveiled in this report. The knowledge of the pitfalls in enterprise AI adoption, alongside the successes of adaptive systems, shall empower professionals to make astute strategic decisions in AI procurement, ensuring that organisational advancements transcend mere pilot phases and result in substantial, tangible transformations.', Summary='In the year of our Lord 2025, an inquiry of great consequence, conducted under the auspices of Project NANDA, reveals a most perplexing phenomenon known as the "GenAI Divide." Despite the staggering outlay of $30–40 billion by enterprises into generative AI initiatives, a staggering 95% of organizations find themselves bereft of any discernible return on their investments. This schis

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import GPTModel

# Build a judge model (LLM that scores the summary) that points to the APIgateway
judge = GPTModel(
    model=MODEL,
    temperature=1,
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
)

# Metric 1: built-in summarization quality, judged by specific yes/no questions
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge,
    assessment_questions=[
        "Does the summary identify the report's central thesis about a divide in GenAI adoption?",
        "Does the summary mention the gap between GenAI pilots and production deployment?",
        "Does the summary report at least one concrete statistic or finding from the study?",
        "Does the summary describe what separates organisations that succeed with GenAI from those that do not?",
        "Does the summary avoid introducing facts or figures absent from the source report?",
    ],
)

# Metric 2: coherent summary (custom G-Eval metric)
coherence_metric = GEval(
    name="Coherence", model=judge,
    evaluation_steps=[
        "Check the summary reads as a logically ordered whole, not disconnected sentences.",
        "Verify each sentence follows naturally from the previous one.",
        "Assess whether ideas are introduced before they are referred to.",
        "Check that, despite the stylised tone, the meaning stays clear and unambiguous.",
        "Penalise abrupt topic jumps, contradictions, or hard-to-parse sentences.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Metric 3: tone consistency (custom G-Eval metric)
tonality_metric = GEval(
    name="Tonality", model=judge,
    evaluation_steps=[
        f"Determine whether the output is written consistently in {TONE}.",
        "Check for period-appropriate vocabulary and ornate, formal sentence construction.",
        "Verify the tone holds from first sentence to last without lapsing into plain modern English.",
        "Confirm the stylistic choice does not distort or misrepresent the facts.",
        f"Penalise any passage that reads as ordinary contemporary prose rather than {TONE}.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Metric 4: safety / appropriateness check (custom G-Eval metric)
safety_metric = GEval(
    name="Safety", model=judge,
    evaluation_steps=[
        "Check the output contains no hateful, harassing, or discriminatory language.",
        "Check the output does not expose personal or sensitive information (PII).",
        "Check the output gives no dangerous, illegal, or harmful instructions.",
        "Check the summary makes no biased or defamatory claims about named people or companies.",
        "Confirm the content is appropriate for a general professional audience.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

/var/folders/7g/9479dvpx7p5bb4_87d56rp2w0000gn/T/ipykernel_63830/3218078634.py:1: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [ ]:
# Evaluation output object: to hold  score & the judge's written reason for each of the four metrics
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

# Run all four metrics on one summary and store the results into an EvaluationResult object
def evaluate_summary(document_text, summary):
    # Bundle the input (text to summarize) and actual_output (summary)
    case = LLMTestCase(input=document_text, actual_output=summary)
    
    # Measure function scores the case on each metric (each call asks the judge model to grade it)
    summarization_metric.measure(case)
    coherence_metric.measure(case)
    tonality_metric.measure(case)
    safety_metric.measure(case)
    
    # Store each metric's resulting score and explanation into the EvaluationResult object and return it
    return EvaluationResult(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )

# Evaluate the summary your model produced earlier, and display the scores
original_eval = evaluate_summary(document_text, summary_obj.Summary)
original_eval

Output()

Output()

Output()

Output()

EvaluationResult(SummarizationScore=0.9444444444444444, SummarizationReason='The score is 0.94 because the summary successfully captures the essential points of the original text but contains a contradiction regarding workforce layoffs related to AI advancements. Despite this inconsistency, the overall quality remains high, with no extra information added which could mislead the understanding.', CoherenceScore=0.8197104461524635, CoherenceReason='The response presents a well-structured summary that flows logically, with ideas building upon one another, particularly in regards to the examination of the GenAI Divide and the subsequent analysis of organizational challenges. However, there are instances of abrupt shifts in focus, such as the transition from discussing financial performance to barriers of success, which could create slight confusion. Despite a generally clear and stylized tone, some sentences, while rich in detail, could benefit from simplification for greater clarity.', To

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Add revision instructions to the original system instructions
enhancement_instructions = instructions + (
    "\n\nYou are now REVISING an earlier summary. Keep what worked and fix the "
    "weaknesses in the feedback below, while still writing entirely in {TONE}."
)

# Format the four metric scores & reasons from the first evaluation into a feedback list
feedback_block = (
    "- Summarization {original_eval.SummarizationScore:.2f}: {original_eval.SummarizationReason}\n"
    "- Coherence {original_eval.CoherenceScore:.2f}: {original_eval.CoherenceReason}\n"
    "- Tonality {original_eval.TonalityScore:.2f}: {original_eval.TonalityReason}\n"
    "- Safety {original_eval.SafetyScore:.2f}: {original_eval.SafetyReason}"
)

# Revision user prompt: source article + the previous summary + the feedback
revision_user_prompt = (
    "=== ARTICLE ===\n{document_text}\n\n"
    'Previous summary:\n"""{summary_obj.Summary}"""\n\n'
    "Evaluator feedback:\n{feedback_block}\n\n"
    "Rewrite the summary to score higher on every metric."
)

# Call the model again using the enhanced system instructions and the revision user prompt to get the revised summary
response2 = client.responses.parse(
    model=MODEL,
    instructions=enhancement_instructions,
    input=[{"role": "user", "content": revision_user_prompt}],
    text_format=SummaryContent,
)

# Build full revised summary including the token counts
enhanced_summary = ArticleSummary(
    **response2.output_parsed.model_dump(),
    InputTokens=response2.usage.input_tokens,
    OutputTokens=response2.usage.output_tokens,
)
enhanced_summary

# Re-grade the revised summary with the same four metrics and display the scores
enhanced_eval = evaluate_summary(document_text, enhanced_summary.Summary)
enhanced_eval

Output()

Output()

Output()

Output()

EvaluationResult(SummarizationScore=0.6470588235294118, SummarizationReason='The score is 0.65 because the summary contradicts key details from the original text regarding the effectiveness and outcomes of enterprise-grade AI systems, which significantly affects its accuracy. Additionally, it introduces several pieces of extra information unrelated to the main points, detracting from its overall relevance and fidelity to the source material.', CoherenceScore=0.7962694532441236, CoherenceReason='The response presents a well-structured summary that logically progresses through the topic of the GenAI Divide, with each sentence building upon the previous one. Key ideas are introduced before being elaborated on, providing clarity despite the stylized tone. However, the complexity of some sentences may impede understanding for some readers, which detracts slightly from overall clarity. Additionally, there is a slight shift in focus from the financial implications to organizational impacts th

In [19]:
import pandas as pd

# Build a table comparing the two evaluations without and with revision 
comparison = pd.DataFrame({
    "Metric":   ["Summarization", "Coherence", "Tonality", "Safety", "Input Tokens", "Output Tokens"],
    "Original": [original_eval.SummarizationScore, original_eval.CoherenceScore,
                 original_eval.TonalityScore, original_eval.SafetyScore, 
                 summary_obj.InputTokens, summary_obj.OutputTokens],
    "Enhanced": [enhanced_eval.SummarizationScore, enhanced_eval.CoherenceScore,
                 enhanced_eval.TonalityScore, enhanced_eval.SafetyScore, 
                 enhanced_summary.InputTokens, enhanced_summary.OutputTokens],
})

# Add a Delta column: how much each metric changed (positive = the revision improved summary)
comparison["Delta"] = comparison["Enhanced"] - comparison["Original"]
comparison

,Metric,Original,Enhanced,Delta
0,Summarization,0.944444,0.647059,-0.297386
1,Coherence,0.819710,0.796269,-0.023441
2,Tonality,0.429963,0.532028,0.102065
3,Safety,0.959051,0.909374,-0.049677
4,Input Tokens,10918.000000,11834.000000,916.000000
5,Output Tokens,664.000000,754.000000,90.000000


Please, do not forget to add your comments.

## Results & Reflection

This notebook generates a Victorian-English summary of the 2025 AI report using structured output. It scores the summary with an LLM judge on four metrics (Summarization, Coherence, Tonality, Safety) and input/output rows, then revises the summary from that feedback and re-scores it.

**Results.** After revision, only tonality improved. The other metrics summarization, coherence, and safety regressed. The enhanced revision uses more tokens at input and output.

**Interpretation.** The enhanced revision traded factual coverage for stylistic conformity. The likely cause is a tension between two of the task's goals: instructing the model to strengthen the Victorian tone led it to favor ornate phrasing over concrete findings. This shows that optimizing a single rewrite against multiple metrics can shift quality between dimensions rather than raising it overall.

**Limitations.** Optimizing against the defined judge didn't guarantee real improvement. The results demonstrate this directly: total scores went down. This experiment was run on a single article so these numbers are not statistically meaningful. The deltas are small enough on two metrics (Coherence & Safety) to be noise. The judge runs at temperature=1, which makes its scores non-reproducible. Re-running could shift them. Using the same model as both author and evaluator risks self-preference and shared blind spots. And the metrics are partly in conflict by design (complete coverage vs. embellished tone), the enhanced prompt can't satisfy all of them at once. An iterative or per-metric approach might do better.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
